# Unitree G1: GPU RL pipeline (MJX + custom PPO)

Settings: **Accelerator = GPU T4 x2**, **Internet = On**, **Persistence = Files**.

Every session: run sections 1–2, then whichever phase cells you need. Cells marked ⏱ <5 min are interactive checks; cells marked 🌙 are long jobs to run with **Save Version → Save & Run All**.

## 1. Get / refresh the code
Picks the **highest-version** copy of the package found under `/kaggle/input` and copies it over `/kaggle/working/g1_rl` (keeping `outputs/`).

**When uploading a new version of the dataset:** in *New Version*, delete the old `g1_rl.zip` before adding the new one, so the dataset holds a single copy. Then click ⟳ next to the dataset in the Input panel.

In [ ]:
import os, re, shutil, glob, subprocess
GIT_URL = ""   # optional: "https://github.com/<you>/g1_rl.git" instead of a dataset
WORK = "/kaggle/working/g1_rl"

def pkg_version(root):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(f"{root}/g1_rl/__init__.py").read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)

if GIT_URL:
    if os.path.exists(f"{WORK}/.git"):
        subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, WORK], check=True)
else:
    roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
             if 'name = "g1_rl"' in open(p).read()]
    assert roots, "Attach the g1_rl dataset or set GIT_URL"
    roots.sort(key=pkg_version, reverse=True)
    for r in roots:
        print(f"found {'.'.join(map(str, pkg_version(r)))}  {r}")
    if len(roots) > 1 and pkg_version(roots[0]) == pkg_version(roots[1]):
        raise SystemExit("Two copies with the same version: delete the old zip from the dataset (see note above).")
    src = roots[0]
    # Replace the code but keep outputs/ (checkpoints, logs, videos) from earlier sessions.
    for item in os.listdir(WORK) if os.path.isdir(WORK) else []:
        if item not in ("outputs",):
            p = os.path.join(WORK, item)
            shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
    shutil.copytree(src, WORK, dirs_exist_ok=True)
    print("using", src)
os.chdir(WORK)
import importlib.util
spec = importlib.util.spec_from_file_location("v", "g1_rl/__init__.py"); v = importlib.util.module_from_spec(spec); spec.loader.exec_module(v)
print("g1_rl version", v.__version__, "|", sorted(os.listdir()))

## 2. Install GPU JAX + MuJoCo/MJX ⏱ ~2 min

In [ ]:
!bash scripts/install_kaggle.sh

---
## Phase 0: setup check & benchmark (done)

In [ ]:
# ⏱ ~2 min
# !python scripts/check_setup.py --platform gpu

In [ ]:
# 🌙 ~15 min
# !python scripts/benchmark.py --platform gpu --scaling --envs 1024,2048,4096,8192 --out outputs/benchmark.json

---
## Phase 1: environment
### Smoke test ⏱ ~3–4 min
512 robots with random actions, 60-step episodes and fast command changes. It prints PASS/FAIL checks and saves `outputs/phase1_smoke/` (report.json, trajectory, video).

In [ ]:
!python scripts/env_smoke.py --platform gpu

In [ ]:
from IPython.display import Video
Video("outputs/phase1_smoke/traj_env0.mp4", embed=True, width=480)

### Environment unit tests ⏱ ~4 min

In [ ]:
!pip install -q pytest && G1RL_TEST_PLATFORM=gpu python -m pytest -q tests/test_env.py

### Full test suite 🌙 ~8 min

In [ ]:
# !G1RL_TEST_PLATFORM=gpu python -m pytest -q

---
## Next phases
Rewards (2), domain randomization (3), PPO training (4), evaluation (5) and export (6) cells are added as each phase lands.